# การพยากรณ์ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7 รายเดือน ด้วย Linear Regression

สมุดโน้ตนี้สร้างแบบจำลอง **Linear Regression** เพื่อพยากรณ์ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7 รายเดือนระดับประเทศ
จากกรมธุรกิจพลังงาน (DOEB) ครอบคลุมช่วงปี พ.ศ. 2560–2567 เฉพาะผู้ค้าน้ำมันตามมาตรา 7

**วัตถุประสงค์หลัก**
1. เตรียมข้อมูลรายเดือนระดับประเทศสำหรับ ADO B7 พร้อมสร้างตัวแปรต้น (Trend Term + One-Hot Month Dummies)
2. แบ่งชุดข้อมูลเป็น Train/Test ตามลำดับเวลา (Time-based split)
3. ฝึกโมเดล Linear Regression และประเมินผลด้วย RMSE และ R²
4. แสดงผลเปรียบเทียบค่าจริงกับค่าพยากรณ์ และวิเคราะห์ค่าความคลาดเคลื่อน (Residuals)

**แหล่งข้อมูล:** กรมธุรกิจพลังงาน — [data.doeb.go.th](https://data.doeb.go.th/dataset/ce709041-d37a-401f-b855-99493816e7b7)
**ไฟล์ที่ใช้:** `data/processed/doeb_ado_b7_combined.csv`

## ขั้นตอนที่ 0: นำเข้าไลบรารีและการตั้งค่าเบื้องต้น

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

TEMPLATE    = 'plotly_white'
FONT_FAMILY = 'Sarabun, IBM Plex Sans Thai, sans-serif'
UNIT_LABEL  = 'พันลิตรต่อเดือน'
DATA_PATH   = '../data/processed/doeb_ado_b7_combined.csv'

print('Libraries loaded successfully.')
print(f'pandas  version : {pd.__version__}')
print(f'numpy   version : {np.__version__}')
import sklearn; print(f'sklearn version : {sklearn.__version__}')
import plotly;  print(f'plotly  version : {plotly.__version__}')

Libraries loaded successfully.
pandas  version : 2.3.3
numpy   version : 2.2.6
sklearn version : 1.7.2
plotly  version : 6.8.0


## ส่วนที่ 1: กรมธุรกิจพลังงาน — ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7 (ADO B7)

**หน่วยงาน:** กรมธุรกิจพลังงาน (DOEB) — กระทรวงพลังงาน
**ชุดข้อมูล:** ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7 รายจังหวัด พ.ศ. 2560–2568
**แหล่งที่มา:** [gdcatalog.go.th/dataset/gdpublish-ado-b7](https://gdcatalog.go.th/dataset/gdpublish-ado-b7)
ขอบเขต: เฉพาะผู้ค้าน้ำมันตามมาตรา 7 — ข้อมูลรายจังหวัด หน่วยวัด: พันลิตรต่อเดือน

In [2]:
try:
    df_raw = pd.read_csv(DATA_PATH, encoding='utf-8', low_memory=False)
except UnicodeDecodeError:
    df_raw = pd.read_csv(DATA_PATH, encoding='cp874', low_memory=False)

print(f'จำนวนแถวทั้งหมด : {len(df_raw):,}')
print(f'จำนวนคอลัมน์   : {df_raw.shape[1]}')
print()
print('ตัวอย่างข้อมูล 5 แถวแรก:')
display(df_raw.head())
print()
print('สรุปชนิดข้อมูลและค่า null:')
display(df_raw.dtypes.to_frame('dtype').join(
    df_raw.isnull().sum().to_frame('null_count')
))

จำนวนแถวทั้งหมด : 125,983
จำนวนคอลัมน์   : 14

ตัวอย่างข้อมูล 5 แถวแรก:


,SUBJECT,YEAR_ID,MONTH_ID,REGION_ZONE_NAME,PROVINCE_FULLNAME,ENTREPRENEUR,ENTRE_SECTIONID,OIL_NAME_ENG,OIL_NAME_THAI,QTY,UNIT,DATAOWNER,year_be,BUSCUSTOMER_SHORT_TH
0,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,พีทีจี เอ็นเนอยี,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
1,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,ไออาร์พีซี,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
2,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,เชฟรอน (ไทย),ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,565.53,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
3,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,ไทยออยล์,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
4,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,ปตท. บริหารธุรกิจค้าปลีก,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN



สรุปชนิดข้อมูลและค่า null:


,dtype,null_count
SUBJECT,object,0
YEAR_ID,int64,0
MONTH_ID,int64,0
REGION_ZONE_NAME,object,0
PROVINCE_FULLNAME,object,0
ENTREPRENEUR,object,0
ENTRE_SECTIONID,object,11564
OIL_NAME_ENG,object,11564
OIL_NAME_THAI,object,11564
QTY,float64,0


### การทำความสะอาดและเตรียมข้อมูลสำหรับแบบจำลอง

**ปัญหาที่พบในชุดข้อมูล:**
- คอลัมน์ `OIL_NAME_ENG` มีค่า `NaN` จำนวนหนึ่ง สำหรับแถวที่ไม่มี `ENTRE_SECTIONID` (ไม่ใช่มาตรา 7)
- ชุดข้อมูลมีน้ำมัน 2 ประเภท ได้แก่ ADO B7 และ ADO ซึ่งต้องกรองเฉพาะ ADO B7
- ค่า `YEAR_ID` อยู่ในรูปปี พ.ศ. ต้องแปลงเป็น ค.ศ. ก่อนสร้าง datetime

**วิธีการเตรียมข้อมูลและสร้างตัวแปรต้น:**
1. กรองเฉพาะแถวที่มี `OIL_NAME_ENG = 'ADO B7'`
2. Aggregate ปริมาณ `QTY` ระดับประเทศรายเดือน (สรุปจากทุกจังหวัดและผู้ประกอบการ)
3. สร้าง **trend** = ลำดับจำนวนเต็มเริ่มจาก 0 แทนแนวโน้มเชิงเส้น
4. สร้าง **one-hot month dummies** สำหรับเดือน 2–12 (เดือน 1 = มกราคม ใช้เป็น base)
5. แบ่ง Train/Test ตามลำดับเวลา: 20 เดือนหลังสุดเป็นชุดทดสอบ (Time-based split)

In [3]:
# ---- 1. กรอง ADO B7 และแถวที่มี OIL_NAME_ENG ----
df = df_raw.dropna(subset=['OIL_NAME_ENG']).copy()
df = df[df['OIL_NAME_ENG'] == 'ADO B7'].copy()
print(f'แถวที่ใช้ (ADO B7 เท่านั้น): {len(df):,}')

# ---- 2. แปลง YEAR_ID (พ.ศ.) เป็น ค.ศ. และสร้าง datetime ----
df['year_ce'] = df['YEAR_ID'] - 543
df['date'] = pd.to_datetime(
    df['year_ce'].astype(str) + '-' + df['MONTH_ID'].astype(str).str.zfill(2) + '-01'
)

# ---- 3. Aggregate: ปริมาณรวมระดับประเทศรายเดือน ----
df_monthly = (
    df.groupby(['date', 'MONTH_ID'], as_index=False)['QTY']
    .sum()
    .sort_values('date')
    .reset_index(drop=True)
)
print(f'จำนวนเดือน (observations): {len(df_monthly)}')
print(f'ช่วงเวลา: {df_monthly["date"].min().strftime("%Y-%m")} — {df_monthly["date"].max().strftime("%Y-%m")}')

# ---- 4. สร้าง Trend term ----
df_monthly['trend'] = np.arange(len(df_monthly))

# ---- 5. สร้าง One-hot month dummies (drop month 1 = January as base) ----
month_dummies = pd.get_dummies(df_monthly['MONTH_ID'], prefix='month', drop_first=True).astype(int)
df_feat = pd.concat([df_monthly[['date', 'QTY', 'trend']], month_dummies], axis=1)

FEATURE_COLS = [c for c in df_feat.columns if c not in ['date', 'QTY']]
print(f'\nตัวแปรต้นทั้งหมด ({len(FEATURE_COLS)} ตัว):')
print(FEATURE_COLS)

# ---- 6. Train/Test split (time-based: 20 เดือนหลังสุดเป็น test) ----
N_TEST = 20
X = df_feat[FEATURE_COLS].values.astype(float)
y = df_feat['QTY'].values.astype(float)
dates_all = df_feat['date'].values

X_train, X_test = X[:-N_TEST], X[-N_TEST:]
y_train, y_test = y[:-N_TEST], y[-N_TEST:]
dates_train = dates_all[:-N_TEST]
dates_test  = dates_all[-N_TEST:]

print(f'\nTrain: {len(X_train)} เดือน  '
      f'({pd.Timestamp(dates_train[0]).strftime("%Y-%m")} — '
      f'{pd.Timestamp(dates_train[-1]).strftime("%Y-%m")})')
print(f'Test : {len(X_test)} เดือน  '
      f'({pd.Timestamp(dates_test[0]).strftime("%Y-%m")} — '
      f'{pd.Timestamp(dates_test[-1]).strftime("%Y-%m")})')

# ---- 7. ฝึกโมเดล Linear Regression ----
model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_pred_test))
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test,  y_pred_test)

print('\n' + '='*55)
print('  ผลการประเมินโมเดล Linear Regression')
print('='*55)
print(f'  Features : trend + one-hot month (เดือน 2–12)')
print(f'  Train/Test split : time-based (last {N_TEST} months = test)')
print(f'  Target   : QTY รวมระดับประเทศรายเดือน (ADO B7)')
print('  ' + '-'*51)
print(f'  Train RMSE : {rmse_train:>14,.2f}  {UNIT_LABEL}')
print(f'  Test  RMSE : {rmse_test:>14,.2f}  {UNIT_LABEL}')
print(f'  Train R²   : {r2_train:>14.4f}')
print(f'  Test  R²   : {r2_test:>14.4f}')
print('='*55)
print()
print('ข้อจำกัดของแบบจำลอง:')
print('  1. ข้อมูลเป็นระดับรวม (Aggregate) ทั่วประเทศ ไม่ใช่ระดับจังหวัดหรือผู้ประกอบการ')
print('  2. ขอบเขตเฉพาะผู้ค้าน้ำมันตามมาตรา 7 เท่านั้น')
print('  3. โมเดลเชิงเส้นไม่สามารถจับรูปแบบที่ไม่เป็นเชิงเส้นได้ (เช่น ผลกระทบ COVID-19)')
print('  4. สมมติว่ารูปแบบฤดูกาล (Seasonality) คงที่ตลอดทุกปี')
print('  5. ไม่มีตัวแปรภายนอก เช่น ราคาน้ำมัน อัตราการเจริญเติบโตทางเศรษฐกิจ นโยบายพลังงาน')

print('\nค่าสัมประสิทธิ์ (Coefficients):')
coef_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Coefficient': model.coef_})
print(f'  Intercept : {model.intercept_:>14,.2f}')
display(coef_df)

แถวที่ใช้ (ADO B7 เท่านั้น): 100,020


จำนวนเดือน (observations): 84
ช่วงเวลา: 2017-01 — 2023-12

ตัวแปรต้นทั้งหมด (12 ตัว):
['trend', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']

Train: 64 เดือน  (2017-01 — 2022-04)
Test : 20 เดือน  (2022-05 — 2023-12)

  ผลการประเมินโมเดล Linear Regression
  Features : trend + one-hot month (เดือน 2–12)
  Train/Test split : time-based (last 20 months = test)
  Target   : QTY รวมระดับประเทศรายเดือน (ADO B7)
  ---------------------------------------------------
  Train RMSE :     222,987.25  พันลิตรต่อเดือน
  Test  RMSE :     789,611.24  พันลิตรต่อเดือน
  Train R²   :         0.4942
  Test  R²   :       -55.9925

ข้อจำกัดของแบบจำลอง:
  1. ข้อมูลเป็นระดับรวม (Aggregate) ทั่วประเทศ ไม่ใช่ระดับจังหวัดหรือผู้ประกอบการ
  2. ขอบเขตเฉพาะผู้ค้าน้ำมันตามมาตรา 7 เท่านั้น
  3. โมเดลเชิงเส้นไม่สามารถจับรูปแบบที่ไม่เป็นเชิงเส้นได้ (เช่น ผลกระทบ COVID-19)
  4. สมมติว่ารูปแบบฤดูกาล (Seasonality) คงที่ตลอดทุกปี
  5. ไม่มีตัวแปร

,Feature,Coefficient
0,trend,-10469.964883
1,month_2,-60173.156783
2,month_3,87948.266433
3,month_4,-13769.177017
4,month_5,-105591.517100
5,month_6,-163349.034217
6,month_7,-193996.971333
7,month_8,-187059.490450
8,month_9,-244839.639567
9,month_10,-140990.270683


### แผนภาพที่ 1 — เปรียบเทียบค่าจริงกับค่าพยากรณ์ (Actual vs Predicted)

แผนภาพเส้นแสดงปริมาณการจำหน่าย ADO B7 ระดับประเทศรายเดือน เปรียบเทียบระหว่างค่าจริง (Actual) และค่าที่โมเดล Linear Regression พยากรณ์ (Predicted)
แบ่งชัดเจนระหว่างช่วง Train (พื้นหลังขาว) และ Test (พื้นหลังแรเงาสีส้ม)
เพื่อประเมินความสามารถในการพยากรณ์ของโมเดลบนข้อมูลที่ไม่เคยเห็นมาก่อน

In [4]:
dates_train_ts = [pd.Timestamp(d) for d in dates_train]
dates_test_ts  = [pd.Timestamp(d) for d in dates_test]

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=dates_train_ts, y=y_train,
    mode='lines',
    name='Actual (Train)',
    line=dict(color='#1f77b4', width=1.8)
))
fig1.add_trace(go.Scatter(
    x=dates_test_ts, y=y_test,
    mode='lines+markers',
    name='Actual (Test)',
    line=dict(color='#d62728', width=2.2),
    marker=dict(size=6)
))
fig1.add_trace(go.Scatter(
    x=dates_train_ts, y=y_pred_train,
    mode='lines',
    name='Predicted (Train)',
    line=dict(color='#17becf', width=1.8, dash='dash')
))
fig1.add_trace(go.Scatter(
    x=dates_test_ts, y=y_pred_test,
    mode='lines+markers',
    name='Predicted (Test)',
    line=dict(color='#ff7f0e', width=2.2, dash='dash'),
    marker=dict(size=6, symbol='x')
))

fig1.add_vrect(
    x0=dates_test_ts[0], x1=dates_test_ts[-1],
    fillcolor='rgba(255,127,14,0.08)',
    layer='below', line_width=0,
    annotation_text=f'ช่วงทดสอบ (n={N_TEST} เดือน)',
    annotation_position='top left',
    annotation_font=dict(size=11, color='#ff7f0e')
)

fig1.update_layout(
    title=(
        f'Linear Regression: Actual vs Predicted — Monthly ADO B7 Sales Volume<br>'
        f'<sup>Train RMSE={rmse_train:,.0f} | Test RMSE={rmse_test:,.0f} | '
        f'Train R²={r2_train:.4f} | Test R²={r2_test:.4f} ({UNIT_LABEL})</sup>'
    ),
    xaxis_title='Month (CE)',
    yaxis_title=f'Sales Volume ({UNIT_LABEL})',
    template=TEMPLATE,
    font=dict(family=FONT_FAMILY, size=13),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='right', x=1
    ),
    hovermode='x unified',
    yaxis=dict(tickformat=',.0f'),
    margin=dict(t=100, b=120)
)

fig1.add_annotation(
    xref='paper', yref='paper', x=0, y=-0.28,
    text=(
        f'คำอธิบาย: ค่าจริงจาก QTY รวมระดับประเทศรายเดือน (ADO B7 เท่านั้น) '
        f'ตัวแปรต้น: trend (ลำดับ 0…N-1) + one-hot เดือน 2–12 (base = ม.ค.) '
        f'Train/Test split: time-based (test = {N_TEST} เดือนหลังสุด) '
        f'หน่วย: {UNIT_LABEL} | ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 | ที่มา: กรมธุรกิจพลังงาน'
    ),
    showarrow=False,
    font=dict(size=10, color='#555555'),
    align='left'
)

fig1.show()

**ที่มาข้อมูล:** คอลัมน์ `QTY` (ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7)  
**การแปลงข้อมูล:** รวม (Aggregate) ระดับประเทศรายเดือน แยกเฉพาะ ADO B7; สร้างค่าพยากรณ์ด้วย Linear Regression (ตัวแปรต้น: trend + one-hot เดือน 2–12); แบ่ง Train/Test แบบ time-based (test = 20 เดือนหลังสุด)  
**หน่วย:** พันลิตรต่อเดือน | ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 | ที่มา: กรมธุรกิจพลังงาน

### แผนภาพที่ 2 — ค่าความคลาดเคลื่อนบนชุดทดสอบ (Residuals on Test Set)

แผนภาพแท่งแสดงค่าความต่างระหว่างค่าจริงและค่าพยากรณ์ (Residual = Actual − Predicted) บนชุดทดสอบ 20 เดือน
สีเขียวหมายถึงโมเดลพยากรณ์ต่ำกว่าความเป็นจริง สีแดงหมายถึงโมเดลพยากรณ์สูงกว่าความเป็นจริง
ช่วยประเมินทิศทางและขนาดของข้อผิดพลาดในแต่ละเดือน

In [5]:
residuals_test = y_test - y_pred_test
date_labels    = [pd.Timestamp(d).strftime('%Y-%m') for d in dates_test]
bar_colors     = ['#2ca02c' if r >= 0 else '#d62728' for r in residuals_test]

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=date_labels,
    y=residuals_test,
    marker_color=bar_colors,
    name='Residual',
    hovertemplate='Month: %{x}<br>Residual: %{y:,.0f}<extra></extra>'
))

fig2.add_hline(
    y=0,
    line_width=1.5, line_dash='dash', line_color='#555555'
)

mae_test = np.mean(np.abs(residuals_test))

fig2.update_layout(
    title=(
        f'Residuals on Test Set — Linear Regression (ADO B7 Monthly Sales)<br>'
        f'<sup>Test RMSE={rmse_test:,.0f} | Test R²={r2_test:.4f} | '
        f'MAE={mae_test:,.0f} ({UNIT_LABEL})</sup>'
    ),
    xaxis_title='Month (CE)',
    yaxis_title=f'Residual: Actual − Predicted ({UNIT_LABEL})',
    template=TEMPLATE,
    font=dict(family=FONT_FAMILY, size=13),
    showlegend=False,
    hovermode='x unified',
    yaxis=dict(tickformat=',.0f'),
    margin=dict(t=100, b=130)
)

fig2.add_annotation(
    xref='paper', yref='paper', x=0, y=-0.32,
    text=(
        f'คำอธิบาย: Residual = QTY จริง − QTY พยากรณ์ บนชุดทดสอบ {N_TEST} เดือน '
        f'สีเขียว = โมเดลต่ำกว่าจริง, สีแดง = โมเดลสูงกว่าจริง '
        f'ตัวแปรต้น: trend + one-hot เดือน 2–12 '
        f'หน่วย: {UNIT_LABEL} | ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 | ที่มา: กรมธุรกิจพลังงาน'
    ),
    showarrow=False,
    font=dict(size=10, color='#555555'),
    align='left'
)

fig2.show()

**ที่มาข้อมูล:** คอลัมน์ `QTY` (ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7)  
**การแปลงข้อมูล:** คำนวณ Residual = QTY จริง − QTY พยากรณ์ จาก Linear Regression บนชุดทดสอบ 20 เดือน (พ.ค. 2565 — ธ.ค. 2566)  
**หน่วย:** พันลิตรต่อเดือน | ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 | ที่มา: กรมธุรกิจพลังงาน

---

## สรุปผลและแนวคิดเพิ่มเติม (Idea Sparks)

### สรุปผลแบบจำลอง Linear Regression

| เมตริก | Train (64 เดือน) | Test (20 เดือน) |
|--------|-----------------|----------------|
| RMSE (พันลิตร/เดือน) | 222,987 | 789,611 |
| R² | 0.4942 | −55.99 |

**ตัวแปรต้น:** trend (ลำดับ 0–83) + one-hot เดือน 2–12 (base = มกราคม) รวม 12 ตัวแปร
**วิธีแบ่ง Train/Test:** Time-based split — 20 เดือนหลังสุด (พ.ค. 2565 — ธ.ค. 2566) เป็นชุดทดสอบ

**ข้อสังเกตหลัก:**
1. **Train R² = 0.49** — โมเดลอธิบายความแปรปรวนในชุดฝึกได้ประมาณครึ่งหนึ่ง แสดงว่า trend และ month dummies ยังไม่เพียงพอ
2. **Test R² = −55.99** — ค่าลบขนาดใหญ่บ่งชี้ว่าโมเดลพยากรณ์ผิดพลาดมากกว่าการใช้ค่าเฉลี่ยเป็นคำตอบ การ generalize ล้มเหลวบนชุดทดสอบ
3. **สาเหตุหลัก:** ชุดฝึกรวมช่วง COVID-19 (ปี 2563) ที่ยอดตกต่ำผิดปกติ ทำให้ค่าสัมประสิทธิ์ trend มีค่าลบ (−10,470) ซึ่งขัดแย้งกับแนวโน้มการฟื้นตัวในชุดทดสอบ
4. **ค่าสัมประสิทธิ์ December (month_12 = +164,421)** สูงที่สุด สอดคล้องกับฤดูกาลปลายปีที่มียอดจำหน่ายสูง

**ข้อจำกัดที่ต้องระบุ:**
1. ข้อมูลเป็น **Aggregate** ระดับประเทศ — ไม่ใช่ระดับจังหวัดหรือผู้ประกอบการ
2. ขอบเขต **เฉพาะผู้ค้าน้ำมันตามมาตรา 7** เท่านั้น ไม่ครอบคลุมผู้บริโภคโดยตรง
3. โมเดลเชิงเส้นสมมติว่า **Trend เป็นเส้นตรง** และ **Seasonality คงที่ทุกปี** ซึ่งไม่เป็นจริงเมื่อมี Shock ภายนอก เช่น COVID-19
4. ไม่มีตัวแปรภายนอก เช่น **ราคาน้ำมัน, GDP, นโยบายรัฐบาล** ซึ่งส่งผลต่อปริมาณการจำหน่ายจริง

### แนวคิดสำหรับการพัฒนาต่อ

- **SARIMA / Prophet:** ใช้โมเดล Time-series เฉพาะทางเพื่อจับ Seasonality และ Trend แบบอัตโนมัติพร้อมจัดการ Structural Break
- **Gradient Boosting (XGBoost/LightGBM):** เพิ่มตัวแปรภายนอก เช่น ราคาน้ำมันดิบ, ดัชนีอุตสาหกรรม
- **Cross-validation แบบ Time Series (TimeSeriesSplit):** ประเมินประสิทธิภาพโมเดลอย่างเป็นระบบกว่า Hold-out เดียว
- **การพยากรณ์รายจังหวัด:** สร้างโมเดลแยกตาม `PROVINCE_FULLNAME` หรือ `REGION_ZONE_NAME` เพื่อวิเคราะห์เชิงพื้นที่